# Rule-check model: GPU training on Colab (Phase 3)

A thin wrapper over the repository's versioned scripts (`ml/`); it adds no modelling code. On the
**full official ContractNLI splits** it trains `deberta_v3_base` (primary), `modernbert_base`
(backbone ablation) and `deberta_v3_base_unseen4` (4 hypotheses held out of training), evaluates them
against the baselines with DEV-only choices, exports ONNX fp32/int8 with parity checks, measures CPU
latency and int8/fp32 TEST accuracy on this runtime, and packs a **verifiable results bundle** to Drive.

Read `ml/notebooks/README.md` first. In short:
- Runtime -> Change runtime type -> **T4 GPU**.
- Source without secrets: upload a `git bundle` of the reviewed commit to Drive (**Option A**,
  recommended), or store a read-only fine-grained GitHub token as the **Colab secret** `GITHUB_TOKEN`
  (Option B). Never paste a token or API key into a cell.
- Everything durable is written to Drive; training is **resumable**. After a disconnect, rerun
  cells 1-5 and then the interrupted step.

No cell prints, logs or stores a token or API key.

## 1. GPU and Drive

In [ ]:
# ruff: noqa: S101, S603, S607, T201, E402  (notebook: shell calls, prints and asserts are intended)
import json
import os
import pathlib
import shutil
import subprocess

subprocess.run(["nvidia-smi"], check=True)
from google.colab import drive

drive.mount("/content/drive")
RUNS = pathlib.Path("/content/drive/MyDrive/compliance-runs")  # durable outputs
RUNS.mkdir(parents=True, exist_ok=True)
REPO = pathlib.Path("/content/compliance")
print("runs directory:", RUNS)

## 2. Source code (choose ONE option)

**Option A (recommended, no token):** on the development machine, after the reviewed commit exists:
`git bundle create compliance.bundle <branch>` and upload the file to `MyDrive/compliance-runs/`.
**Option B:** add a fine-grained, read-only (Contents: read) GitHub token for this one repository as the
Colab secret `GITHUB_TOKEN`, and set `USE_OPTION_B_GITHUB = True`.
In both cases set `EXPECTED_COMMIT` to the full commit hash to train; the cell refuses anything else.

In [ ]:
EXPECTED_COMMIT = ""  # full 40-character commit hash (required)
BUNDLE = RUNS / "compliance.bundle"  # Option A
USE_OPTION_B_GITHUB = False
GITHUB_REPO = "JCHETAN26/compliance"

assert len(EXPECTED_COMMIT) == 40, "set EXPECTED_COMMIT to the full commit hash"
if not REPO.exists():
    if not USE_OPTION_B_GITHUB:
        # `git bundle verify` needs a repository: verify inside a fresh empty one, then clone.
        check = pathlib.Path("/content/.bundle-check")
        shutil.rmtree(check, ignore_errors=True)
        subprocess.run(["git", "init", "--quiet", str(check)], check=True)
        subprocess.run(["git", "-C", str(check), "bundle", "verify", str(BUNDLE)], check=True)
        shutil.rmtree(check)
        subprocess.run(["git", "clone", "--quiet", str(BUNDLE), str(REPO)], check=True)
    else:
        # The token comes from a Colab secret and reaches git only through the environment of a
        # GIT_ASKPASS helper: never printed, never in a URL, never written to .git/config.
        from google.colab import userdata

        helper = pathlib.Path("/content/.git-askpass.sh")
        helper.write_text(
            '#!/bin/sh\ncase "$1" in Username*) echo x-access-token;; *) echo "$CLONE_TOKEN";; esac\n'
        )
        helper.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(helper),
            "GIT_TERMINAL_PROMPT": "0",
            "CLONE_TOKEN": userdata.get("GITHUB_TOKEN"),
        }
        try:
            subprocess.run(
                ["git", "clone", "--quiet", f"https://github.com/{GITHUB_REPO}.git", str(REPO)],
                check=True,
                env=env,
            )
        finally:
            helper.unlink(missing_ok=True)
            del env
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", EXPECTED_COMMIT], check=True)
head = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPO), "status", "--porcelain"], capture_output=True, text=True, check=True
).stdout
assert head == EXPECTED_COMMIT, head
assert not dirty, dirty
print("training source:", head, "(clean)")
os.chdir(REPO)

## 3. Environment: the locked dependencies + CUDA PyTorch of the same version

`uv sync --frozen` installs exactly `uv.lock` (Python 3.11, CPU torch 2.14.1); the torch wheel is then
replaced by the **CUDA 12.6 build of the same version** and the ML tool pins are added. All later
commands call `.venv/bin/python` directly (never `uv run` or `make`, which would re-sync the CPU
torch), and `assert_cuda()` re-checks the build right before every GPU step.

In [ ]:
subprocess.run(["pip", "install", "--quiet", "uv==0.8.22"], check=True)
subprocess.run(["uv", "sync", "--frozen"], check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall-package",
        "torch",
        "torch==2.14.1+cu126",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ],
    check=True,
)
subprocess.run(
    ["uv", "pip", "install", "--python", ".venv/bin/python", "-r", "ml/requirements-tools.txt"],
    check=True,
)
PY = str(REPO / ".venv/bin/python")


def assert_cuda():
    out = subprocess.run(
        [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.split()
    assert out[0] == "2.14.1+cu126", f"wrong torch build {out[0]} (was the environment re-synced?)"
    assert out[1] == "True", "CUDA is not available"


def run(*args, **env):
    """Run a repository script with the project interpreter (output streamed)."""
    subprocess.run([PY, "-m", *args], check=True, env={**os.environ, "PYTHONPATH": "", **env})


assert_cuda()
print("torch 2.14.1+cu126 with CUDA ready")

## 4. Pinned artifacts (ContractNLI archive + BGE encoder; SHA-256 verified)

In [ ]:
run("compliance_ingestion.embeddings", "fetch", ARTIFACT_CACHE_DIR=".cache")
run("compliance_seed.fetch", ARTIFACT_CACHE_DIR=".cache")

## 5. Processed data (official splits, segments, candidate ranking) and classic baselines

Prepared once and kept on Drive, so every session and every run uses the **same `data_version`** (a
resumed run refuses a different one). The dense candidate ranking uses the pinned BGE encoder on GPU.
The majority and TF-IDF baselines run on CPU on the same data.

In [ ]:
DATA_DRIVE = RUNS / "data-full"
DATA = REPO / ".cache/ml/data/full"
if DATA_DRIVE.exists():
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.copytree(DATA_DRIVE, DATA)
    print("restored data from Drive")
else:
    assert_cuda()
    run("ml.data.prepare", "--device", "cuda", "--threads", "2")
    shutil.copytree(DATA, DATA_DRIVE)
subprocess.run(
    [
        PY,
        "-c",
        "from pathlib import Path; from ml.data.store import load_dataset; "
        "load_dataset(Path('.cache/ml/data/full'))",
    ],
    check=True,
)  # verifies every file hash
manifest = json.loads((DATA / "manifest.json").read_text())
print("data_version", manifest["data_version"], manifest["counts"])
BASELINES = RUNS / "baselines"
if not (BASELINES / "classic_baselines.json").exists():
    run("ml.baselines.classic", "--out", str(BASELINES))

## 6. Training (resumable: rerun the cell after a disconnect)

Each run selects its checkpoint/epoch and label top-k on DEV only; TEST is predicted after that selection is frozen.
Checkpoints stay on Drive (several GB per run; see the README for the space plan).

In [ ]:
assert_cuda()
run(
    "ml.train.train",
    "--config",
    "ml/train/configs/deberta_v3_base.yaml",
    "--run-dir",
    str(RUNS / "deberta_v3_base"),
)

In [ ]:
assert_cuda()
run(
    "ml.train.train",
    "--config",
    "ml/train/configs/modernbert_base.yaml",
    "--run-dir",
    str(RUNS / "modernbert_base"),
)

In [ ]:
assert_cuda()
run(
    "ml.train.train",
    "--config",
    "ml/train/configs/deberta_v3_base_unseen4.yaml",
    "--run-dir",
    str(RUNS / "deberta_v3_base_unseen4"),
)

## 7. Evaluation, export, CPU latency and quantized TEST accuracy

The serving backbone is chosen on DEV. Export writes ONNX fp32 + int8 with parity checks to Drive. The
benchmark measures this runtime's CPU (recorded in the result file). The int8/fp32 TEST accuracy run
is required for the phase and takes a while on Colab's CPU.

In [ ]:
def evaluate(*extra):
    run(
        "ml.eval.evaluate",
        "--run-dir",
        str(RUNS / "deberta_v3_base"),
        "--ablation-run-dir",
        str(RUNS / "modernbert_base"),
        "--unseen-run-dir",
        str(RUNS / "deberta_v3_base_unseen4"),
        "--baselines-dir",
        str(BASELINES),
        *extra,
    )


evaluate()
params = json.loads(pathlib.Path("ml/eval/results/inference_params.json").read_text())
CHOSEN = pathlib.Path(params["run_dir"])
ARTIFACT = RUNS / f"artifact-{CHOSEN.name}"
assert_cuda()
run(
    "ml.export.export",
    "--run-dir",
    str(CHOSEN),
    "--params",
    "ml/eval/results/inference_params.json",
    "--model-eval",
    "ml/eval/results/model_eval.json",
    "--out",
    str(ARTIFACT),
)
THREADS = str(os.cpu_count() or 2)
run(
    "ml.export.bench",
    "--artifact-dir",
    str(ARTIFACT),
    "--run-dir",
    str(CHOSEN),
    "--label",
    "colab_cpu",
    "--threads",
    THREADS,
    "--out",
    "ml/eval/results/latency_colab_cpu.json",
)
run(
    "ml.export.eval_onnx",
    "--artifact-dir",
    str(ARTIFACT),
    "--run-dir",
    str(CHOSEN),
    "--split",
    "test",
    "--threads",
    THREADS,
    "--out",
    "ml/eval/results/quantization_eval.json",
)

## 8. Optional: the Anthropic zero-shot baseline here (real API calls, billed to your key)

Leave `RUN_LLM = False` to run it locally after import instead (`make baseline-llm`). If enabled, the key
is read from the Colab secret `ANTHROPIC_API_KEY` into the child process's environment only. The dry
runs print the prompt and a rough cost estimate; re-running resumes without paying twice.

In [ ]:
RUN_LLM = False
if RUN_LLM:
    from google.colab import userdata

    for split in ("dev", "test"):
        run("ml.baselines.llm", "--split", split, "--dry-run")
    key_env = {"ANTHROPIC_API_KEY": userdata.get("ANTHROPIC_API_KEY")}
    for split in ("dev", "test"):
        run("ml.baselines.llm", "--split", split, "--out", str(BASELINES), **key_env)
    del key_env

## 9. Final evaluation and artifact parameters

Re-evaluates with the measured latency (and the LLM results if present: the cascade tau is then
selected on DEV), and refreshes the artifact manifest so the served parameters and version match the
final DEV selections (weights unchanged).

In [ ]:
evaluate("--bench", "ml/eval/results/latency_colab_cpu.json")
final = json.loads(pathlib.Path("ml/eval/results/inference_params.json").read_text())
assert final["run_dir"] == params["run_dir"], "the DEV backbone choice changed; re-run step 7"
run(
    "ml.export.export",
    "--run-dir",
    str(CHOSEN),
    "--params",
    "ml/eval/results/inference_params.json",
    "--model-eval",
    "ml/eval/results/model_eval.json",
    "--out",
    str(ARTIFACT),
    "--refresh-manifest",
)

## 10. Results bundle for the repository (int8 artifact only; no training weights, no secrets)

In [ ]:
BUNDLE_OUT = RUNS / f"results-bundle-{EXPECTED_COMMIT[:12]}.tar"
args = [
    "ml.colab.bundle",
    "pack",
    "--out",
    str(BUNDLE_OUT),
    "--data",
    str(DATA),
    "--baselines",
    str(BASELINES),
    "--results",
    "ml/eval/results",
    "--artifact",
    str(ARTIFACT),
]
for name in ("deberta_v3_base", "modernbert_base", "deberta_v3_base_unseen4"):
    args += ["--run", f"{name}={RUNS / name}"]
run(*args)
crit = json.loads(pathlib.Path("ml/eval/results/baselines_comparison.json").read_text())[
    "handoff_criterion"
]
print("handoff criterion (model vs TF-IDF macro-F1 on TEST):", crit)
print("download", BUNDLE_OUT, "then run locally: make import-colab-bundle BUNDLE=<path>")